# 02 — Enriquecimento dos artigos SciELO

Este notebook lê os arquivos coletados no passo 01 e entra em cada link de artigo para enriquecer metadados.

## Entrada padrão

```text
data/01_search/scielo_*_search_results.csv
```

## Saída padrão

```text
data/02_enriched/02_scielo_articles_<keyword_slug>_enriched.csv
data/02_enriched/02_scielo_articles_combined_enriched.csv
logs/checkpoints/
```

In [ ]:
# Instalação, caso precise:
# !pip install beautifulsoup4 pandas requests tqdm lxml unidecode

In [ ]:
import ast
import json
import re
import time
from pathlib import Path

import pandas as pd
from pandas.errors import EmptyDataError
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from unidecode import unidecode


## 1. Configuração geral

In [ ]:
def find_project_dir(start=None):
    """
    Encontra a raiz do projeto, mesmo quando o notebook está dentro de:
    Scielo/01.data_collect, Scielo/02.data_enrich, Scielo/03.data_cleaning etc.

    Critério:
    - procura uma pasta ancestral que contenha README.md/.gitignore e a pasta Scielo.
    - se não encontrar, usa o diretório atual.
    """
    current = Path(start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        has_project_marker = (
            (candidate / 'README.md').exists()
            or (candidate / '.gitignore').exists()
            or (candidate / 'requirements.txt').exists()
        )
        has_scielo_folder = (candidate / 'Scielo').exists()

        if has_project_marker and has_scielo_folder:
            return candidate

    return current


PROJECT_DIR = find_project_dir()

# Diretórios padrão do projeto. Na sua estrutura, data/ e logs/ ficam na raiz do projeto,
# não dentro de Scielo/02.data_enrich.
DATA_DIR = PROJECT_DIR / 'data'
LOG_DIR = PROJECT_DIR / 'logs'

SEARCH_DIR = DATA_DIR / '01_search'
ENRICHED_DIR = DATA_DIR / '02_enriched'
CHECKPOINT_DIR = LOG_DIR / 'checkpoints'

ENRICHED_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Use None para processar todos os arquivos de busca encontrados.
# Ou informe uma lista específica de slugs, por exemplo: ['elites', 'classes_dominantes'].
KEYWORD_SLUGS = None

SLEEP_SECONDS = 0.3
CHECKPOINT_EVERY = 50
REQUEST_TIMEOUT = 40
MAX_RETRIES = 2

print('CWD:', Path.cwd().resolve())
print('PROJECT_DIR:', PROJECT_DIR)
print('SEARCH_DIR:', SEARCH_DIR)
print('ENRICHED_DIR:', ENRICHED_DIR)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)


## 2. Funções auxiliares

In [ ]:
def clean_text(value):
    """Normaliza espaços e trata vazios."""
    if value is None or pd.isna(value):
        return ''

    return re.sub(r'\s+', ' ', str(value)).strip()


def normalize_doi(value):
    """Normaliza DOI removendo prefixos comuns."""
    if value is None or pd.isna(value):
        return ''

    value = str(value).strip()
    value = value.replace('DOI:', '')
    value = value.replace('doi:', '')
    value = value.replace('https://doi.org/', '')
    value = value.replace('http://dx.doi.org/', '')
    return value.strip()


def safe_literal_list(value):
    """Converte string de lista em lista Python."""
    if isinstance(value, list):
        return value

    if value is None or pd.isna(value):
        return []

    value = str(value).strip()

    if value in ['', '[]', 'nan', 'None']:
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [clean_text(x) for x in parsed if clean_text(x)]
        return [clean_text(parsed)] if clean_text(parsed) else []
    except Exception:
        return [clean_text(value)] if clean_text(value) else []


def get_meta_content(soup, name):
    """Busca uma meta tag por name ou property."""
    tag = soup.find('meta', attrs={'name': name})
    if tag and tag.get('content'):
        return clean_text(tag.get('content'))

    tag = soup.find('meta', attrs={'property': name})
    if tag and tag.get('content'):
        return clean_text(tag.get('content'))

    return ''


def get_all_meta_content(soup, name):
    """Busca múltiplas meta tags de mesmo nome."""
    tags = soup.find_all('meta', attrs={'name': name})
    values = [clean_text(tag.get('content')) for tag in tags if tag.get('content')]
    return list(dict.fromkeys(values))

## 3. Extração de resumo, referências e instituições brutas

In [ ]:
def extract_abstract_from_page(soup):
    """Extrai resumo por meta tag e, se necessário, por blocos textuais."""
    candidates = [
        get_meta_content(soup, 'citation_abstract'),
        get_meta_content(soup, 'description'),
        get_meta_content(soup, 'og:description'),
    ]

    for candidate in candidates:
        if candidate and len(candidate) > 80:
            return candidate

    text_blocks = []

    for tag in soup.find_all(['p', 'div', 'section']):
        text = clean_text(tag.get_text(' ', strip=True))
        if re.search(r'\b(resumo|abstract)\b', unidecode(text.lower())) and len(text) > 100:
            text_blocks.append(text)

    return max(text_blocks, key=len) if text_blocks else ''


def extract_references_from_page(soup):
    """Extrai referências em formato bruto. É uma heurística exploratória."""
    refs = []

    selectors = [
        '.ref', '.references', '.ref-list li', 'li.ref', 'div.ref', 'p.ref',
        '[id*=ref]', '[class*=ref]'
    ]

    for selector in selectors:
        for tag in soup.select(selector):
            text = clean_text(tag.get_text(' ', strip=True))
            if len(text) > 30:
                refs.append(text)

    refs = list(dict.fromkeys(refs))
    if refs:
        return refs

    full_text = clean_text(soup.get_text(' ', strip=True))
    norm_text = unidecode(full_text.lower())

    positions = [
        norm_text.find(marker)
        for marker in ['referencias', 'references', 'bibliografia']
        if norm_text.find(marker) != -1
    ]

    if not positions:
        return []

    start = min(positions)
    ref_text = full_text[start:start + 25000]

    rough_refs = re.split(
        r'(?<=\.)\s+(?=[A-ZÁÉÍÓÚÂÊÔÃÕÇ][A-ZÁÉÍÓÚÂÊÔÃÕÇa-záéíóúâêôãõç\-]+,\s)',
        ref_text,
    )

    return [clean_text(ref) for ref in rough_refs if len(clean_text(ref)) > 40]


def extract_raw_institutions_from_page(soup):
    """Tenta capturar afiliações/instituições brutas no HTML."""
    institutions = get_all_meta_content(soup, 'citation_author_institution')

    # Fallbacks comuns em páginas SciELO/JATS renderizadas.
    selectors = [
        '.aff', '.affiliation', '.author-affiliation',
        '[class*=aff]', '[id*=aff]',
        '[class*=institution]', '[id*=institution]'
    ]

    for selector in selectors:
        for tag in soup.select(selector):
            text = clean_text(tag.get_text(' ', strip=True))
            if len(text) > 5:
                institutions.append(text)

    return list(dict.fromkeys(institutions))

## 4. Extração de metadados por artigo

In [ ]:
def request_with_retries(session, url, max_retries=2, timeout=40):
    """Faz requisição com retry simples."""
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/124.0 Safari/537.36'
        )
    }

    last_error = None

    for attempt in range(max_retries + 1):
        try:
            response = session.get(url, headers=headers, timeout=timeout)
            response.raise_for_status()
            return response
        except Exception as exc:
            last_error = exc
            time.sleep(1 + attempt)

    raise last_error


def extract_article_metadata(url, session=None):
    """Extrai metadados de uma página de artigo SciELO."""
    if session is None:
        session = requests.Session()

    response = request_with_retries(
        session=session,
        url=url,
        max_retries=MAX_RETRIES,
        timeout=REQUEST_TIMEOUT,
    )

    soup = BeautifulSoup(response.text, 'lxml')

    title = get_meta_content(soup, 'citation_title')
    authors = get_all_meta_content(soup, 'citation_author')
    institutions = extract_raw_institutions_from_page(soup)
    keywords = get_all_meta_content(soup, 'citation_keywords')

    journal = get_meta_content(soup, 'citation_journal_title')
    publication_date = get_meta_content(soup, 'citation_publication_date')
    doi = normalize_doi(get_meta_content(soup, 'citation_doi'))
    language = get_meta_content(soup, 'citation_language')
    publisher = get_meta_content(soup, 'citation_publisher')

    abstract = extract_abstract_from_page(soup)
    references = extract_references_from_page(soup)
    full_text = clean_text(soup.get_text(' ', strip=True))

    return {
        'article_title': title,
        'article_authors': authors,
        'article_institutions': institutions,
        'article_keywords': keywords,
        'article_journal': journal,
        'article_publication_date': publication_date,
        'article_doi': doi,
        'article_language': language,
        'article_publisher': publisher,
        'article_abstract': abstract,
        'article_references': references,
        'article_references_count': len(references),
        'article_full_text_sample': full_text[:50000],
        'article_scrape_error': '',
    }

## 5. Leitura dos arquivos do passo 01

In [ ]:
def get_keyword_slug_from_search_file(path):
    """
    Extrai o slug da keyword a partir de diferentes padrões de nome.

    Aceita:
    - scielo_elites_search_results.csv
    - search_results_elites.csv
    - elites_search_results.csv
    """
    name = path.name

    if name.startswith('scielo_') and name.endswith('_search_results.csv'):
        return name.replace('scielo_', '').replace('_search_results.csv', '')

    if name.startswith('search_results_') and name.endswith('.csv'):
        return name.replace('search_results_', '').replace('.csv', '')

    if name.endswith('_search_results.csv'):
        return name.replace('_search_results.csv', '')

    return path.stem


def find_search_files():
    """
    Busca arquivos de resultado do passo 01 em locais prováveis.

    A busca é deliberadamente defensiva porque a estrutura pode variar entre:
    - data/01_search/
    - data/
    - raiz do projeto
    - Scielo/data/01_search/
    """
    candidate_dirs = [
        SEARCH_DIR,
        DATA_DIR,
        PROJECT_DIR,
        PROJECT_DIR / 'Scielo' / 'data' / '01_search',
        PROJECT_DIR / 'Scielo' / 'data',
    ]

    patterns = [
        'scielo_*_search_results.csv',
        'search_results_*.csv',
        '*_search_results.csv',
    ]

    ignore_terms = [
        'combined',
        'enriched',
        'checkpoint',
        'clean',
        'analytical',
        'analysis',
    ]

    files = []

    for directory in candidate_dirs:
        if not directory.exists():
            continue

        for pattern in patterns:
            files.extend(directory.glob(pattern))

    # Deduplica preservando ordem
    unique_files = []
    seen = set()

    for path in files:
        resolved = path.resolve()

        if resolved in seen:
            continue

        name_lower = path.name.lower()

        if any(term in name_lower for term in ignore_terms):
            continue

        seen.add(resolved)
        unique_files.append(path)

    return sorted(unique_files)


search_files = find_search_files()

if KEYWORD_SLUGS is not None:
    allowed = set(KEYWORD_SLUGS)
    search_files = [
        path for path in search_files
        if get_keyword_slug_from_search_file(path) in allowed
    ]

print(f'Arquivos de busca encontrados: {len(search_files)}')

for path in search_files:
    size_kb = path.stat().st_size / 1024
    print(f'- {path} | {size_kb:.1f} KB | slug={get_keyword_slug_from_search_file(path)}')

if not search_files:
    raise FileNotFoundError(
        'Nenhum arquivo de busca foi encontrado. '
        'Verifique se o passo 01 salvou arquivos em data/01_search/ ou se PROJECT_DIR está correto.'
    )


## 6. Enriquecimento por keyword

In [ ]:
def load_search_file(path):
    """
    Carrega resultado de busca e normaliza colunas básicas.

    Retorna None quando:
    - arquivo está vazio;
    - CSV não tem colunas;
    - CSV não tem linhas;
    - coluna link não existe;
    - coluna link existe, mas todos os links estão vazios.
    """
    keyword_slug = get_keyword_slug_from_search_file(path)

    if not path.exists():
        print(f'[SKIP] arquivo inexistente: {path}')
        return None

    if path.stat().st_size == 0:
        print(f'[SKIP] arquivo vazio: {path.name}')
        return None

    try:
        df = pd.read_csv(path)
    except EmptyDataError:
        print(f'[SKIP] CSV sem colunas: {path.name}')
        return None
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding='utf-8-sig')
    except Exception as exc:
        print(f'[SKIP] erro ao ler {path.name}: {exc}')
        return None

    if df.empty:
        print(f'[SKIP] CSV sem linhas: {path.name}')
        return None

    # Normaliza possíveis nomes alternativos para a coluna de link
    link_candidates = ['link', 'url', 'article_url', 'article_link']

    if 'link' not in df.columns:
        for candidate in link_candidates:
            if candidate in df.columns:
                df = df.rename(columns={candidate: 'link'})
                break

    if 'link' not in df.columns:
        print(f'[SKIP] CSV sem coluna obrigatória link: {path.name} | colunas={list(df.columns)}')
        return None

    df['link'] = df['link'].fillna('').astype(str).str.strip()
    df = df.query("link != ''").drop_duplicates(subset=['link']).reset_index(drop=True)

    if df.empty:
        print(f'[SKIP] CSV sem links válidos: {path.name}')
        return None

    if 'authors' in df.columns:
        df['authors'] = df['authors'].apply(safe_literal_list)

    if 'search_keyword_slug' not in df.columns:
        df['search_keyword_slug'] = keyword_slug

    if 'search_keyword' not in df.columns:
        df['search_keyword'] = df['search_keyword_slug'].astype(str).str.replace('_', ' ', regex=False)

    if 'search_file' not in df.columns:
        df['search_file'] = path.name

    print(f'[OK] {path.name}: {len(df)} links válidos')
    return df


def enrich_search_df(df, keyword_slug, sleep_seconds=0.3, checkpoint_every=50):
    """Enriquece todos os artigos de uma base de busca."""
    session = requests.Session()
    records = []
    links = df['link'].fillna('').tolist()

    for idx, link in enumerate(tqdm(links, total=len(links), desc=keyword_slug)):
        try:
            if not link:
                metadata = {'article_scrape_error': 'missing link'}
            else:
                metadata = extract_article_metadata(link, session=session)
        except Exception as exc:
            metadata = {'article_scrape_error': str(exc)}

        records.append(metadata)

        if (idx + 1) % checkpoint_every == 0:
            checkpoint_df = pd.concat(
                [df.iloc[:idx + 1].reset_index(drop=True), pd.DataFrame(records).reset_index(drop=True)],
                axis=1,
            )
            checkpoint_df.to_csv(
                CHECKPOINT_DIR / f'checkpoint_scielo_{keyword_slug}_enriched.csv',
                index=False,
                encoding='utf-8-sig',
            )

        time.sleep(sleep_seconds)

    metadata_df = pd.DataFrame(records)

    enriched_df = pd.concat(
        [df.reset_index(drop=True), metadata_df.reset_index(drop=True)],
        axis=1,
    )

    output_path = ENRICHED_DIR / f'02_scielo_articles_{keyword_slug}_enriched.csv'

    enriched_df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f'[SAVE] {output_path} | linhas={len(enriched_df)}')

    return enriched_df


In [ ]:
all_enriched = []
skipped_files = []

for search_file in search_files:
    keyword_slug = get_keyword_slug_from_search_file(search_file)
    search_df = load_search_file(search_file)

    if search_df is None:
        skipped_files.append({
            'file': str(search_file),
            'reason': 'empty_or_invalid_search_file'
        })
        continue

    enriched_df = enrich_search_df(
        df=search_df,
        keyword_slug=keyword_slug,
        sleep_seconds=SLEEP_SECONDS,
        checkpoint_every=CHECKPOINT_EVERY,
    )

    if not enriched_df.empty:
        all_enriched.append(enriched_df)

if all_enriched:
    combined_enriched = pd.concat(all_enriched, ignore_index=True)
else:
    combined_enriched = pd.DataFrame()

combined_path = ENRICHED_DIR / '02_scielo_articles_combined_enriched.csv'
combined_enriched.to_csv(combined_path, index=False, encoding='utf-8-sig')

skipped_path = LOG_DIR / 'skipped_search_files.csv'
pd.DataFrame(skipped_files).to_csv(skipped_path, index=False, encoding='utf-8-sig')

print(f'[SAVE] {combined_path} | linhas={len(combined_enriched)}')
print(f'[SAVE] {skipped_path} | ignorados={len(skipped_files)}')

print(combined_enriched.shape)
combined_enriched.head()


## 7. Diagnóstico do enriquecimento

In [ ]:
if combined_enriched.empty:
    print('Nenhum artigo enriquecido.')
else:
    diagnostic_enrichment = (
        combined_enriched
        .assign(
            has_article_title=lambda x: x['article_title'].fillna('').ne('') if 'article_title' in x else False,
            has_article_doi=lambda x: x['article_doi'].fillna('').ne('') if 'article_doi' in x else False,
            has_article_abstract=lambda x: x['article_abstract'].fillna('').ne('') if 'article_abstract' in x else False,
            has_error=lambda x: x['article_scrape_error'].fillna('').ne('') if 'article_scrape_error' in x else False,
        )
        .groupby('search_keyword', as_index=False)
        .agg(
            records=('title', 'count'),
            titles=('has_article_title', 'sum'),
            dois=('has_article_doi', 'sum'),
            abstracts=('has_article_abstract', 'sum'),
            errors=('has_error', 'sum'),
        )
        .sort_values('records', ascending=False)
    )

    diagnostic_enrichment.to_csv(ENRICHED_DIR / 'diagnostic_enrichment_by_keyword.csv', index=False, encoding='utf-8-sig')
    display(diagnostic_enrichment)